# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/darksider747/flyrank-1st/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

Method: Random Forest

I'm choosing Random Forest because I already have real evidence it
performs well on this exact type of data - the starter pipeline's own
results showed Random Forest beating both Logistic Regression and
Decision Tree on Precision@50 (0.740 vs 0.400 and 0.540), and beating
the hand-written baseline rule by roughly 3x (0.740 vs 0.240).

Conceptually, Random Forest combines many different decision trees
(each trained on a slightly different slice of data), then combines
their votes into a final prediction. This tends to be more reliable
than a single decision tree, which can latch onto quirky patterns
specific to the data it happened to see. Since real declining pages
likely decline for different combinations of reasons (low CTR,
staleness, position drops), a method that can weigh many signals
together fits this problem better than a simple rule with only
AND/OR logic - which is exactly the "why ML beats a fixed rule"
reasoning from Week 2.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess

REPO_URL = "https://github.com/darksider747/flyrank-1st"
REPO_DIR = "/content/flyrank-1st"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")

df["is_declining"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining"].value_counts())

Now in: /content/flyrank-1st
Loaded 30000 rows
is_declining
1    16262
0    13738
Name: count, dtype: int64


## 2. Split design

Split design: client_holdout (grouped by client)

Instead of randomly splitting individual pages between train and test,
I split by entire clients - all pages from a given client go either
fully into training or fully into testing, never both.

This matters because if pages from the same client appeared in both
sets, the model could "cheat" by learning that client's specific
quirks during training, then get an artificially easy win recognizing
those same quirks during testing. A client_holdout split forces the
model to be tested on genuinely new websites it has never seen
anything from - a fair test of whether it learned a real, general
pattern, not just memorized one client's specific behavior. This
matches the same validation approach the starter pipeline itself used.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

# Rebuild the baseline score, so it can be compared on the same split later
tier_avg_ctr = df.groupby("position_tier")["ctr"].transform("mean")
df["ctr_below_tier"] = df["ctr"] < (0.7 * tier_avg_ctr)
df["is_stale"] = df["days_since_last_update"] >= 91
df["baseline_score"] = df["ctr_below_tier"].astype(int) + df["is_stale"].astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print(f"Train: {len(train_df)} rows, {train_df['client_id'].nunique()} clients")
print(f"Test: {len(test_df)} rows, {test_df['client_id'].nunique()} clients")

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Clients in both train and test (should be 0): {len(overlap)}")

Train: 22885 rows, 24 clients
Test: 7115 rows, 8 clients
Clients in both train and test (should be 0): 0


## 3. Train + compare vs my baseline

Extended comparison across multiple K values (same test set, same metric):

| K   | Baseline Precision@K | Model Precision@K |
|-----|----------------------|--------------------|
| 10  | 0.800                | 0.800              |
| 20  | 0.550                | 0.750              |
| 50  | 0.500                | 0.780              |
| 100 | 0.470                | 0.690              |

An honest observation: at K=10, the model does NOT beat the baseline -
they're tied. The model's real advantage shows up at K=20 and beyond,
where it pulls clearly ahead (0.750-0.780 vs 0.470-0.550).

This makes sense given a limitation I noted in Week 4: my baseline
score only has 3 possible values (0, 1, 2), so thousands of pages tie
at the same score with no real ordering beyond that. It can identify
the most obvious top candidates well, but has no way to meaningfully
rank anyone past that. The model produces a smooth probability for
every page, so it can keep making meaningful distinctions much further
down the ranked list - which is exactly where its advantage over the
baseline grows.

In [11]:
from sklearn.ensemble import RandomForestClassifier

feature_cols = ["ctr", "avg_position", "days_since_last_update", "impressions_90d", "word_count"]

train_X = train_df[feature_cols].fillna(train_df[feature_cols].median())
train_y = train_df["is_declining"]
test_X = test_df[feature_cols].fillna(train_df[feature_cols].median())
test_y = test_df["is_declining"]

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(train_X, train_y)

print("Model trained on", len(train_X), "rows")

test_df["model_prob"] = model.predict_proba(test_X)[:, 1]

def precision_at_k(df_scored, score_col, label_col, k=50):
    top_k = df_scored.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

model_p50 = precision_at_k(test_df, "model_prob", "is_declining", k=50)
baseline_p50 = precision_at_k(test_df, "baseline_score", "is_declining", k=50)

print(f"Baseline Precision@50: {baseline_p50:.3f} (~{round(baseline_p50*50)} of top 50 correct)")
print(f"Model Precision@50:    {model_p50:.3f} (~{round(model_p50*50)} of top 50 correct)")
for k in [10, 20, 50, 100]:
    baseline_pk = precision_at_k(test_df, "baseline_score", "is_declining", k=k)
    model_pk = precision_at_k(test_df, "model_prob", "is_declining", k=k)
    print(f"K={k:3d} | Baseline: {baseline_pk:.3f} | Model: {model_pk:.3f}")

Model trained on 22885 rows
Baseline Precision@50: 0.500 (~25 of top 50 correct)
Model Precision@50:    0.780 (~39 of top 50 correct)
K= 10 | Baseline: 0.800 | Model: 0.800
K= 20 | Baseline: 0.550 | Model: 0.750
K= 50 | Baseline: 0.500 | Model: 0.780
K=100 | Baseline: 0.470 | Model: 0.690


## 4. Errors and interpretation

Feature importance:
impressions_90d (0.326), avg_position (0.290), and word_count (0.201)
were the model's most influential features. Notably, days_since_last_update
(0.055) was the LEAST important - barely used at all. This lines up with
my Week 4 finding that staleness was only a MIXED signal (not cleanly
confirmed), and that my top-20 review noticed a suspicious identical
value (104 days) across many rows, suggesting a possible data quality
issue. The model appears to have independently "discovered" that this
signal isn't very reliable, which supports my earlier caution about it.

Overall accuracy was only 0.568 - barely better than a coin flip. This
looks concerning at first, but is not actually contradictory with the
strong Precision@50 result (0.780): accuracy measures every single
prediction equally, while Precision@K specifically measures whether
the TOP-ranked pages are correct - which is what actually matters for
a real reviewer, who only checks the top of the list, not every page.

Error analysis: I looked at "confident false positives" - cases where
the model was very sure (80%+) a page was declining, but it actually
wasn't. There were 555 such cases. These pages tend to have low CTR
(median 0.0) but only moderate position and relatively fresh content
(median 20 days since update, not stale). This suggests the model may
be over-relying on low CTR as a decline signal on its own, even when
other signals (freshness, position) suggest the page is otherwise fine -
a real limitation worth flagging for anyone using this model's output.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)
test_df["predicted_declining"] = (test_df["model_prob"] >= 0.5).astype(int)
test_df["correct"] = test_df["predicted_declining"] == test_df["is_declining"]

print(f"Overall accuracy: {test_df['correct'].mean():.3f}")

# Look at confident wrong predictions specifically
confident_wrong = test_df[(test_df["model_prob"] >= 0.8) & (test_df["is_declining"] == 0)]
print(f"\nConfident false positives (model very sure it's declining, but it's not): {len(confident_wrong)}")
print(confident_wrong[["ctr", "avg_position", "days_since_last_update", "impressions_90d"]].describe())

impressions_90d           0.325846
avg_position              0.290311
word_count                0.200571
ctr                       0.128172
days_since_last_update    0.055099
dtype: float64
Overall accuracy: 0.568

Confident false positives (model very sure it's declining, but it's not): 555
              ctr  avg_position  days_since_last_update  impressions_90d
count  555.000000    555.000000              555.000000       555.000000
mean     0.125982     16.256396               36.349550      1917.218018
std      0.228476     10.821504               38.220956      3923.847088
min      0.000000      1.800000                4.000000         5.000000
25%      0.000000      8.200000               13.000000       134.500000
50%      0.000000     12.700000               20.000000       832.000000
75%      0.170000     21.400000               25.000000      2230.000000
max      2.630000     61.100000              183.000000     54464.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.